# Cleaning Dataset for CNN
The dataset present in `/datasets/raw` is the raw dataset. This code shows every step for all data to be processed and cleaned till it is kept inside `/final_dataset`.
Multiple modules will be used in this process. Each module will be imported and shortly documented. The final documentations will be present in the `/README.md` file.

<b>Copies files from input_dir to output_dir with optional include/exclude filters.</b>

Rules:
- If includes is provided, only files containing any include string (case-insensitive) are copied.
- If excludes is provided, files containing any exclude string (case-insensitive) are skipped.
- If skip is provided, files will be skipped based on skip no.
- A unique suffix is added to each copied file name to avoid collisions.

using custom module `copy_filtered`

In [1]:
from modules.copy_filtered import copy_filtered

Making array of input, output, includes, excludes, skip and seed for normal dataset conditions

In [ ]:
parameters = [
    ("./datasets/raw/actions/train/hit", "./datasets/classified_data/hit", None, None, 15, None),
    ("./datasets/raw/actions/train/kick", "./datasets/classified_data/kick", None, None, 15, None),
    ("./datasets/raw/actions/train/punch", "./datasets/classified_data/punch", None, None, 15, None),
    ("./datasets/raw/actions/train/stand", "./datasets/classified_data/stand", None, None, 20, None),
    ("./datasets/raw/actions/train/wave", "./datasets/classified_data/wave", None, None, 30, None),
    ("./datasets/raw/actions/train/shoot_gun", "./datasets/classified_data/gun", None, None, 50, None),
    ("./datasets/raw/actions/test/hit", "./datasets/classified_data/hit", None, None, 15, None),
    ("./datasets/raw/actions/test/punch", "./datasets/classified_data/punch", None, None, 8, None),
    ("./datasets/raw/actions/test/kick", "./datasets/classified_data/kick", None, None, 8, None),
    ("./datasets/raw/actions/test/stand", "./datasets/classified_data/stand", None, None, 25, None),
    ("./datasets/raw/actions/test/wave", "./datasets/classified_data/wave", None, None, 40, None),
    ("./datasets/raw/actions/test/shoot_gun", "./datasets/classified_data/gun", None, None, 50, None),
    ("./datasets/raw/Gunns", "./datasets/classified_data/gun", None, None, None, None),
    ("./datasets/raw/people_with_guns", "./datasets/classified_data/gun", None, None, None, None),
    ("./datasets/raw/knife/train", "./datasets/classified_data/knife", None, None, None, None),
    ("./datasets/raw/knife/test", "./datasets/classified_data/knife", None, None, None, None),
    ("./datasets/raw/Knife_Dataset/images", "./datasets/classified_data/knife", None, None, None, None)
]

Using copy_filtered module with given array

In [ ]:
for params in parameters:
    copy_filtered(*params)


Making copy of labeled files in classified_data
- use pandas for reading .csv labels
- use custom module `copy_to_label_dir`

In [ ]:
import pandas as pd
from modules.copy_to_label_dir import copy_to_label_dir

In [ ]:
training = pd.read_csv("./datasets/raw/HAR/Training_set.csv")

maps={
    "sitting":"sit",
    "standing":"stand",
    "sleeping":"sleep",
    "dancing":"dance",
    "hugging":"hug"
}

training = training[training["label"].isin(maps.keys())]
training["label"] = training["label"].replace(maps)
training.head()

In [ ]:
for i, row in training.iterrows():
    copy_to_label_dir(
        filename=row["filename"],
        input_dir="./datasets/raw/HAR/train",
        output_dir="./datasets/classified_data",
        label=row["label"]
    )

Extract frames from videos in input_dir.

    Parameters:
    - fps: number of frames per second to extract (not dependent on video fps)
    - start: float (0 to 1), start portion of video
    - end: float (0 to 1), end portion of video

    Output:
    - Saves extracted frames as .png in output_dir with unique names

Using custom module `frame_extractor`.

In [ ]:
from modules.frame_extractor import frame_extractor

In [ ]:
frame_extractor(
    input_dir = "./datasets/raw/Fight/Images",
    output_dir = "./datasets/classified_data/fight",
    fps = 2,
    start = 0.0,
    end = 1.0
)

Show file counts in each directory using custom module `file_counter`

In [3]:
from modules.file_counter import count_files_per_dir

In [4]:
dirs = count_files_per_dir("./datasets/classified_data")
dirs

{'./datasets/classified_data': 0,
 './datasets/classified_data\\dance': 840,
 './datasets/classified_data\\fight': 109054,
 './datasets/classified_data\\gun': 338,
 './datasets/classified_data\\hit': 277,
 './datasets/classified_data\\hug': 840,
 './datasets/classified_data\\kick': 429,
 './datasets/classified_data\\knife': 614,
 './datasets/classified_data\\punch': 607,
 './datasets/classified_data\\sit': 840,
 './datasets/classified_data\\sleep': 840,
 './datasets/classified_data\\stand': 555,
 './datasets/classified_data\\wave': 259}

<b>Make new final classes dataset:</b>
* Make into three classes:
    - Fight
    - Armed
    - Other
* Other will have merged of:
    - sit
    - stand
    - sleep
    - hug

In [5]:
armed_params = [    
    ("./datasets/classified_data/gun", "./datasets/final_classes/Armed", None, None, None, None),
    ("./datasets/classified_data/knife", "./datasets/final_classes/Armed", None, None, None, None),
]

for params in armed_params:
    copy_filtered(*params)

armed_dirs = count_files_per_dir("./datasets/final_classes/")
armed_dirs

Copied 338/338 filtered files to './datasets/final_classes/Armed'
Copied 614/614 filtered files to './datasets/final_classes/Armed'


{'./datasets/final_classes/': 0, './datasets/final_classes/Armed': 952}

In [7]:
other_params = [
    ("./datasets/classified_data/stand", "./datasets/final_classes/Other", None, None, 3, 42),
    ("./datasets/classified_data/sit", "./datasets/final_classes/Other", None, None, 3, 42),
    ("./datasets/classified_data/sleep", "./datasets/final_classes/Other", None, None, 3, 42),
    ("./datasets/classified_data/hug", "./datasets/final_classes/Other", None, None, 3, 42),
]


for params in other_params:
    copy_filtered(*params)

other_dirs = count_files_per_dir("./datasets/final_classes/")
other_dirs

Copied 185/555 filtered files to './datasets/final_classes/Other'
Copied 280/840 filtered files to './datasets/final_classes/Other'
Copied 280/840 filtered files to './datasets/final_classes/Other'
Copied 280/840 filtered files to './datasets/final_classes/Other'


{'./datasets/final_classes/': 0,
 './datasets/final_classes/Armed': 952,
 './datasets/final_classes/Other': 1025}

In [9]:
fight_params = ("./datasets/classified_data/fight", "./datasets/final_classes/Fight", None, None, 105, 42)
copy_filtered(*fight_params)
fight_dirs = count_files_per_dir("./datasets/final_classes/")
fight_dirs

Copied 1049/110093 filtered files to './datasets/final_classes/Fight'


{'./datasets/final_classes/': 0,
 './datasets/final_classes/Armed': 952,
 './datasets/final_classes/Fight': 1049,
 './datasets/final_classes/Other': 1025}